# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [5]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = "gpt-5-mini" #'gpt-4o-mini'
openai = OpenAI()

API key looks good so far


In [6]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [7]:
ed = Website("https://edwarddonner.com")
ed.links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/',
 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/',
 'https://edwarddonner.com/2025/04/21/the-complete-agentic-ai-engineering-course/',
 'https://edwarddonner.com/2025/04/21/the-

## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [8]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [9]:
print(link_system_prompt)

You are provided with a list of links found on a webpage. You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}



In [10]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [11]:
print(get_links_user_prompt(ed))

Here is the list of links on the website of https://edwarddonner.com - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/
https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/
https://edwarddo

In [12]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [13]:
# Anthropic has made their site harder to scrape, so I'm using HuggingFace..

huggingface = Website("https://huggingface.co")
huggingface.links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/Qwen/Qwen-Image-Edit',
 '/deepseek-ai/DeepSeek-V3.1-Base',
 '/xai-org/grok-2',
 '/deepseek-ai/DeepSeek-V3.1',
 '/ByteDance-Seed/Seed-OSS-36B-Instruct',
 '/models',
 '/spaces/Qwen/Qwen-Image-Edit',
 '/spaces/enzostvs/deepsite',
 '/spaces/zerogpu-aoti/wan2-2-fp8da-aoti-faster',
 '/spaces/lvwerra/jupyter-agent-2',
 '/spaces/NXN-Labs/Voost',
 '/spaces',
 '/datasets/fka/awesome-chatgpt-prompts',
 '/datasets/nvidia/Granary',
 '/datasets/nvidia/Llama-Nemotron-VLM-Dataset-v1',
 '/datasets/nvidia/Nemotron-CC-v2',
 '/datasets/nvidia/Nemotron-Post-Training-Dataset-v2',
 '/datasets',
 '/join',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/allenai',
 '/facebook',
 '/amazon',
 '/google',
 '/Intel',
 '/microsoft',
 '/grammarly',
 '/Writer',
 '/docs/

In [14]:
get_links("https://huggingface.co")

{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'brand assets', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'join / sign up', 'url': 'https://huggingface.co/join'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'products - models', 'url': 'https://huggingface.co/models'},
  {'type': 'products - datasets', 'url': 'https://huggingface.co/datasets'},
  {'type': 'products - spaces', 'url': 'https://huggingface.co/spaces'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs'},
  {'type': 'learn / tutorials', 'url': 'https://huggingface.co/learn'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'c

In [15]:
get_links("https://anthropic.com")

{'links': [{'type': 'company page',
   'url': 'https://www.anthropic.com/company'},
  {'type': 'team page', 'url': 'https://www.anthropic.com/team'},
  {'type': 'careers page', 'url': 'https://www.anthropic.com/careers'},
  {'type': 'jobs page', 'url': 'https://www.anthropic.com/jobs'},
  {'type': 'contact / sales',
   'url': 'https://www.anthropic.com/contact-sales'},
  {'type': 'customers / case studies',
   'url': 'https://www.anthropic.com/customers'},
  {'type': 'product page', 'url': 'https://www.anthropic.com/claude'},
  {'type': 'product page', 'url': 'https://www.anthropic.com/claude-code'},
  {'type': 'product page', 'url': 'https://www.anthropic.com/max'},
  {'type': 'product platform', 'url': 'https://claude.ai/'},
  {'type': 'enterprise page', 'url': 'https://www.anthropic.com/enterprise'},
  {'type': 'api overview', 'url': 'https://www.anthropic.com/api'},
  {'type': 'api docs', 'url': 'https://docs.anthropic.com/'},
  {'type': 'pricing page', 'url': 'https://www.anthropi

## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [16]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [17]:
print(get_all_details("https://huggingface.co"))

Found links: {'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'}, {'type': 'company page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'about / brand guidelines', 'url': 'https://huggingface.co/brand'}, {'type': 'enterprise', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing', 'url': 'https://huggingface.co/pricing'}, {'type': 'careers / jobs (external)', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'products - models', 'url': 'https://huggingface.co/models'}, {'type': 'products - datasets', 'url': 'https://huggingface.co/datasets'}, {'type': 'products - spaces', 'url': 'https://huggingface.co/spaces'}, {'type': 'documentation', 'url': 'https://huggingface.co/docs'}, {'type': 'blog', 'url': 'https://huggingface.co/blog'}, {'type': 'learning resources', 'url': 'https://huggingface.co/learn'}, {'type': 'community forum', 'url': 'https://discuss.huggingface.co'}, {'type': 'community - discord', 'url': 'https://huggingface.co/join/discor

In [26]:
# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."


In [19]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [20]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'}, {'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'models page', 'url': 'https://huggingface.co/models'}, {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'}, {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'endpoints (inference) service', 'url': 'https://endpoints.huggingface.co'}, {'type': 'documentation', 'url': 'https://huggingface.co/docs'}, {'type': 'learn / tutorials', 'url': 'https://huggingface.co/learn'}, {'type': 'brand assets', 'url': 'https://huggingface.co/brand'}, {'type': 'blog', 'url': 'https://huggingface.co/blog'}, {'type': 'careers / jobs (apply)', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'community forum', 'url': 'https://discuss.huggingface.co'}, {'type': 'Disc

'You are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\nHugging Face – The AI community building the future.\nWebpage Contents:\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 1M+ models\nTrending on\nthis week\nModels\nQwen/Qwen-Image-Edit\nUpdated\nabout 16 hours ago\n•\n40.4k\n•\n1.31k\ndeepseek-ai/DeepSeek-V3.1-Base\nUpdated\n3 days ago\n•\n15.3k\n•\n917\nxai-org/grok-2\nUpdated\n2 days ago\n•\n1.39k\n•\n666\ndeepseek-ai/DeepSeek-V3.1\nUpdated\n3 days ago\n•\n21.4k\n•\n550\nByteDance-Seed/Seed-OSS-36B-Instruct\nUpdated\nabout 9 hours ago\n•\n6.36k\n•\n319\nBrowse 1M+ models\nSpaces\nRunning\non\nZero\

In [21]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [22]:
create_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'}, {'type': 'company page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://huggingface.co/join'}, {'type': 'jobs / applications', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'brand / assets', 'url': 'https://huggingface.co/brand'}, {'type': 'blog / news', 'url': 'https://huggingface.co/blog'}, {'type': 'documentation / resources', 'url': 'https://huggingface.co/docs'}, {'type': 'community forum', 'url': 'https://discuss.huggingface.co'}, {'type': 'status page', 'url': 'https://status.huggingface.co/'}, {'type': 'product - endpoints', 'url': 'https://endpoints.huggingface.co'}, {'type': 'social - GitHub', 'url': 'https://github.com/huggingface'}, {'type': 'social - Twitter', 'url': 'https://twitter.com/huggingface'}, {

Hugging Face — The AI community building the future
==================================================

Tagline
-------
The platform where the machine learning community collaborates on models, datasets, and applications.

Quick overview
--------------
- Host, discover and collaborate on ML models, datasets and applications.
- Massive open hub: 1M+ models, 250k+ datasets, and 400k+ applications/Spaces to explore and reuse.
- Community-first open source tooling that powers research and production ML across modalities (text, image, video, audio, 3D).

What they offer
--------------
- Hub (Models & Datasets): Browse and share models and datasets; build your ML profile and portfolio.
- Spaces: Deploy and share interactive ML apps (zero-to-GPU in a few clicks).
- Compute & Inference: Paid GPU compute and optimized inference endpoints (GPU starting at $0.60/hour).
- Team & Enterprise: Managed team and enterprise plans (starting at $20/user/month) with SSO, regions, priority support, audit logs, resource groups and private dataset viewers.
- Docs & Community: Extensive docs, tutorials, a forum and an active community for collaboration and learning.

Flagship open-source projects
-----------------------------
Hugging Face maintains an ecosystem of widely used ML libraries and tools (examples and community adoption counts shown on their site):
- Transformers — state-of-the-art models for PyTorch (large community adoption)
- Diffusers — diffusion models in PyTorch
- Tokenizers — fast tokenizers optimized for research & production
- Datasets — access & share datasets for any ML task
- PEFT, TRL, Accelerate, Safetensors, Transformers.js, Text Generation Inference, and more

Customers & community
---------------------
- More than 50,000 organizations use Hugging Face, including research labs, enterprises and consumer tech companies.
- Notable organizations on the Hub: Ai2, AI at Meta, Amazon, Google, Intel, Microsoft, Grammarly, Writer and many more.
- The platform supports both public open collaboration and enterprise-grade private workflows.

Company culture (what to expect)
-------------------------------
- Community-driven and open source: the company centers its products and values on collaboration, transparency and shared tooling.
- Developer and researcher focused: tooling and docs geared toward enabling reproducible research and production ML.
- Portfolio & visibility: contributors and teams can build public portfolios and share work with the global ML community.
- Fast-moving and impact-oriented: a place to rapidly ship models, apps and libraries used by researchers and enterprises worldwide.

Careers & joining the team
--------------------------
- Hugging Face lists opportunities via their Jobs page (see the website). They regularly hire across engineering, research, product, community and enterprise roles.
- Working there offers direct contribution to popular open-source projects and products used by millions, plus close collaboration with leading organizations across industry and academia.
- For current openings and application details: visit the Hugging Face Jobs page on their website.

Get started
-----------
- Sign up on Hugging Face to host models, upload datasets and deploy Spaces.
- Explore AI Apps or browse the Hub to find models, datasets and example Spaces.
- Use the docs and community forum for guides, examples and support.
- Enterprise and compute pricing/details are available on the Pricing and Enterprise pages.

Links & resources
-----------------
- Website: huggingface.co
- Community & support: Docs, Forum, Discord
- Social / code: GitHub, Twitter, LinkedIn

Contact / Next steps
--------------------
- Try it: sign up and explore models, datasets and Spaces.
- For teams: review Enterprise features and pricing (SSO, priority support, audit logs).
- For contributors: check the Jobs and Open Source project repos to start contributing.

For more details, visit Hugging Face’s homepage and documentation.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [23]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [24]:
stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'}, {'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'brand assets', 'url': 'https://huggingface.co/brand'}, {'type': 'careers page', 'url': 'https://huggingface.co/join'}, {'type': 'careers (external/jobs)', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'enterprise', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing', 'url': 'https://huggingface.co/pricing'}, {'type': 'blog', 'url': 'https://huggingface.co/blog'}, {'type': 'documentation', 'url': 'https://huggingface.co/docs'}, {'type': 'learning', 'url': 'https://huggingface.co/learn'}, {'type': 'community forum', 'url': 'https://discuss.huggingface.co'}, {'type': 'status', 'url': 'https://status.huggingface.co/'}, {'type': 'product endpoints', 'url': 'https://endpoints.huggingface.co'}, {'type': 'product (chat)', 'url': 'https://huggingface.co/chat'}, {'type': 'GitHub', 'url': 'https://github.com/huggin

# Hugging Face — The AI community building the future

The platform where the machine learning community collaborates on models, datasets, and applications.

---

## At a glance
- Home of machine learning: host, share and collaborate on unlimited public models, datasets and apps.
- Community scale: Browse 1M+ models, 250k+ datasets and 400k+ applications (Spaces).
- Trusted by 50,000+ organizations including Ai2, Meta AI, Amazon, Google, Intel, Microsoft, Grammarly and Writer.
- Open source foundation: widely used projects such as Transformers, Diffusers, Tokenizers, Datasets and many more.

---

## What we offer

### Platform
- Models — Browse and host models across text, image, audio, video and 3D.
- Datasets — Share and access datasets for any ML task.
- Spaces — Deploy and share interactive ML applications (Gradio/Streamlit) with 1-click GPU upgrades.
- Community — Profiles, followers, trending models and collaborative workflows.

### Enterprise & Compute
- Inference Endpoints — optimized serving for production models.
- Managed Compute — GPU-backed compute for training and running Spaces (starting at $0.60 / hour for GPU).
- Team & Enterprise — SSO, regions, priority support, audit logs, resource groups, private dataset viewers. Business pricing from $20/user/month.

### Open source tooling (examples)
- Transformers — state-of-the-art models for PyTorch (148,801).
- Diffusers — diffusion models in PyTorch (30,472).
- Tokenizers — fast tokenization tools (10,019).
- Datasets — access & share datasets (20,551).
- Many more: Safetensors, Hub Python Library, PEFT, Accelerate, Text Generation Inference, Transformers.js, TRL, etc.

---

## Why customers & partners choose Hugging Face
- Collaborative network effect — large community contributions accelerate model and dataset development.
- Production-ready tooling — from research (Transformers, Diffusers) to serving (Inference Endpoints, TGI).
- Multi-modality support — text, vision, audio, video and 3D.
- Enterprise-grade controls — security, access management and dedicated support for teams.

---

## Company culture
- Community-first: built around open source, transparency and collaboration — contributors and users build together.
- Research + Product: supporting both cutting‑edge research and production workflows.
- Portfolio-driven: individuals and organizations can build public portfolios and ML profiles.
- Impact & scale: work used by thousands of orgs and millions of developers worldwide.

---

## Careers & how to join
- Hugging Face hires across engineering, research, product, design, data science, community and enterprise roles.
- Perks of joining: contribute to widely used open-source projects, collaborate with large industry partners, and impact the tooling that powers modern ML.
- Learn about open roles and apply at: https://huggingface.co/jobs (Jobs page linked on the site).

---

## For prospects & investors
- Strong network effects: rapidly growing repository of models, datasets and applications.
- Broad adoption across industry leaders and startups.
- Open-source leadership with highly popular toolkits that lower friction from research to production.
- Monetization through compute, inference, and enterprise subscriptions while supporting an active free community tier.

---

## Get started / Contact
- Explore AI Apps or browse 1M+ models: https://huggingface.co
- Docs & developer resources: https://huggingface.co/docs
- Enterprise inquiries & pricing: https://huggingface.co/pricing
- Community & social: GitHub, Twitter, LinkedIn, Discord (links available on site)

---

Hugging Face — accelerating machine learning by bringing the community, tooling and production solutions together.

In [25]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'company page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'brand guidelines', 'url': 'https://huggingface.co/brand'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'blog', 'url': 'https://huggingface.co/blog'}, {'type': 'documentation', 'url': 'https://huggingface.co/docs'}, {'type': 'community forum', 'url': 'https://discuss.huggingface.co'}, {'type': 'github', 'url': 'https://github.com/huggingface'}, {'type': 'twitter', 'url': 'https://twitter.com/huggingface'}, {'type': 'linkedin', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'status', 'url': 'https://status.huggingface.co/'}, {'type': 'models catalog', 'url': 'https://huggingface.co/models'}, {'type': 'datasets catalog', 'url': 'https://huggingface.co/datasets'}, {'type': 'spaces', 'url': 

# Hugging Face — The AI community building the future

Website: https://huggingface.co

---

Why Hugging Face?
- The collaboration platform where the machine learning community builds, shares and ships models, datasets and applications.
- Community-first and open-source driven: a hub for researchers, engineers and organizations to accelerate ML development across text, image, video, audio and 3D.

Quick snapshot
- 1M+ models on the Hub
- 400k+ applications (Spaces)
- 250k+ datasets
- More than 50,000 organizations using the platform
- Team size listed: ~210 members
- Open-source projects (highlights) with community activity:
  - Transformers — 148,801
  - Diffusers — 30,472
  - Tokenizers — 10,019
  - Datasets — 20,551
  - Text Generation Inference — 10,442
  - PEFT, TRL, Accelerate and many more

Core offerings
- Hugging Face Hub: host, discover and collaborate on unlimited public models, datasets and apps.
- Spaces: deploy ML demos and apps (run on CPU or GPU with a few clicks).
- Models & Datasets: search, reuse and contribute; trending models and datasets are updated continuously.
- Inference Endpoints & Compute: managed inference and GPU-backed compute; GPU pricing starts at $0.60/hour.
- Enterprise: team/enterprise plans with SSO, regions, priority support, audit logs, resource groups and private dataset viewers. Enterprise pricing from $20/user/month.

Customers & community
- Widely adopted across academia, startups and large enterprises. Representative organizations on the Hub include:
  - AI2, Meta AI, Amazon, Google, Intel, Microsoft, Grammarly, Writer
- Activity-driven platform: community members update datasets, publish Spaces, author blog articles and research — the Hub is as much social & collaborative as it is a registry.

Open source foundation
- Hugging Face builds and maintains foundational ML tooling used across research and production:
  - Model libraries (Transformers, Diffusers)
  - Fast tokenizers, serialization (safetensors)
  - Training & deployment tools (Accelerate, TRL, Text Generation Inference)
  - JavaScript and browser-side runtimes (Transformers.js)

Company culture (as presented)
- Community-first and collaborative: the site emphasizes building ML better together, sharing work, and building public portfolios.
- Open-source ethos: much of the tooling and infrastructure is developed in the open and shaped by community contributions.
- Developer- and researcher-friendly: focus on tooling that moves projects faster from research to production.

Careers & joining the team
- Hugging Face posts open roles via its Jobs page (link on the site). The organisation is actively growing (team ~210 members).
- Roles typically span engineering, research, product, developer relations/community and enterprise/customer success (see Jobs page for current openings and details).
- Why join: contribute to widely used open-source tooling, work at the intersection of research and product, and engage with a large, active ML community.

For prospective customers / investors
- Proven platform adoption across thousands of organizations and a rich open-source ecosystem.
- Enterprise capabilities for security, governance and support.
- Multiple monetization vectors: compute/inference billing, enterprise subscriptions and professional services.

Get started / Contact
- Explore the Hub: https://huggingface.co
- Try Spaces and example apps (browse from the Hub)
- Enterprise & pricing: details on the site (Compute starting $0.60/hr GPU; Team & Enterprise from $20/user/month)
- Jobs & careers: see the Jobs link on the site

---

Hugging Face is more than a product — it’s a community and an open-source platform designed to accelerate machine learning collaboration, discovery and deployment.

In [27]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'brand assets', 'url': 'https://huggingface.co/brand'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'documentation', 'url': 'https://huggingface.co/docs'}, {'type': 'blog', 'url': 'https://huggingface.co/blog'}, {'type': 'community forum', 'url': 'https://discuss.huggingface.co'}, {'type': 'github', 'url': 'https://github.com/huggingface'}, {'type': 'twitter', 'url': 'https://twitter.com/huggingface'}, {'type': 'linkedin', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'models page', 'url': 'https://huggingface.co/models'}, {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'}, {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'}, {'type': 'endpoints', 'url': 'http

# Hugging Face — The AI community building the future (and making it friendlier)
Welcome to Hugging Face: the place where models, datasets and apps mingle like a really nerdy cocktail party. If you build, ship, or admire machine learning — you belong here.

---

## Quick facts (the bite-sized brag)
- Home of 1M+ models, 250k+ datasets and 400k+ Spaces (apps) — yes, really.  
- Open-source staples: Transformers (148,803⭐), Diffusers (30,472⭐), Tokenizers, Safetensors, PEFT, TRL, Accelerate, Transformers.js and more.  
- Used by 50,000+ organizations (Ai2, AI at Meta, Amazon, Google, Intel, Microsoft, Grammarly, Writer… you get the idea).  
- Team: ~210+ people and a global, highly active community (55k+ followers on the org profile).  
- Paid offerings: GPU compute from $0.60/hr and Team & Enterprise plans from $20/user/month.

---

## What we actually do (TL;DR)
- Host and collaborate on unlimited public models, datasets and applications.  
- Let you run models in the browser, on GPUs, or in production with Inference Endpoints and Text Generation Inference (TGI).  
- Provide Spaces — tiny web apps that show off your models (and occasionally make the internet laugh).  
- Build and maintain ML tooling so teams move faster: training, tokenization, inference, RL fine-tuning, secure weight formats, and more.

How to start: Sign up → Browse 1M+ models → Spin up a Space or an Endpoint → Brag to your boss.

---

## Products & OSS highlights (aka the cool toolbox)
- Transformers — state-of-the-art models for PyTorch (and friends).  
- Diffusers — SOTA diffusion models for images.  
- Text Generation Inference (TGI) — serve LLMs efficiently.  
- Spaces — deploy demos & apps (Jupyter Agents, image editors, virtual try-ons, etc.).  
- Transformers.js — run models in the browser (no backend sweat required).  
- Tokenizers & Safetensors — speed, safety, and fewer headaches.

If it helps you build ML, we probably either made it, host it, or wrote the docs.

---

## Customers & community (the people who trust us)
More than 50,000 organizations — from research labs to big enterprises and fast-growing startups. Featured users include:
- AI2, AI at Meta, Amazon, Google, Intel, Microsoft, Grammarly, Writer — and thousands more teams, researchers and hobbyists.

The community is the product: contributors, dataset curators, Space creators, and maintainers keep the ecosystem humming.

---

## Culture (what it’s like to work/play here)
- Community-first: open source at heart. Collaboration > ego.  
- Fast-moving: shipping useful tools and models is the daily norm.  
- Low-friction: build, share, repeat. Portfolios and public profiles are encouraged (flex your ML cred).  
- A little playful: we take AI seriously but not ourselves. Expect memes, creative Spaces, and community jokes alongside hard engineering.

---

## Careers & joining us
Want in? We have a Jobs page — and we're often hiring across engineering, research, product, design, community/DevRel, and enterprise teams. If you like open source, shipping tools that thousands use, and occasional nerd humor, check the Jobs page and apply.

Helpful hints for applicants:
- Show us code, models, datasets, or Spaces you actually built. Your public profile speaks volumes.  
- Be community-minded: contributions and collaboration are highly valued.

---

## One-liner for investors
A community-driven, open-source-first ML platform powering discovery, deployment and production of models — used by tens of thousands of orgs and backed by a huge contributor ecosystem. Scalability, monetization (compute + enterprise), and a sticky developer base = repeatable value.

---

Want to try? Sign up, explore AI apps, or browse 1M+ models. Hug a model (metaphorically) — Hugging Face.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>